# What is PL/SQL?

### PL/SQL = Procedural Language / Structured Query Language
### Its Oracle progrramming language that extends SQL by adding proggramming features such as variables, conditions, loops, exception handelling, procedures, functions, packages, and triggers.

# Connection Establishment :

## Path :

In [23]:
path = r'E:\Study\Github\Sec\Scripts\DB Loading\connect_db.py'
exec(open(path, encoding='utf-8').read())

## Connection :

In [34]:
connect_oracle()

✨ Permanent %%plsql registered from secrets file! Session is live.


## Checking : 

In [26]:
%%plsql
DECLARE
    v_msg VARCHAR2(100) := 'Brilliant! Your database setup is officially complete.';
BEGIN
    DBMS_OUTPUT.PUT_LINE(v_msg);
END;

Brilliant! Your database setup is officially complete.


# Data Loading :

## Customer Orders Table : 

### SCHEMA Formatting : 

In [27]:
%%plsql
DECLARE
    CURSOR c_co_tables IS
        SELECT table_name 
        FROM user_tables 
        WHERE table_name IN ('CUSTOMER_ORDERS', 'ORDER_ITEMS', 'ORDERS', 'PRODUCTS', 'CUSTOMERS', 'STORES', 'SHIPMENTS', 'INVENTORY');
BEGIN
    FOR r_tab IN c_co_tables LOOP
        BEGIN
            EXECUTE IMMEDIATE 'DROP TABLE ' || r_tab.table_name || ' CASCADE CONSTRAINTS';
            DBMS_OUTPUT.PUT_LINE('🧹 Dropping empty structural fragment: ' || r_tab.table_name);
        EXCEPTION
            WHEN OTHERS THEN NULL;
        END;
    END LOOP;
    DBMS_OUTPUT.PUT_LINE('✅ Schema successfully reset to pristine condition.');
END;


🧹 Dropping empty structural fragment: CUSTOMERS
🧹 Dropping empty structural fragment: INVENTORY
🧹 Dropping empty structural fragment: ORDERS
🧹 Dropping empty structural fragment: ORDER_ITEMS
🧹 Dropping empty structural fragment: PRODUCTS
🧹 Dropping empty structural fragment: SHIPMENTS
🧹 Dropping empty structural fragment: STORES
✅ Schema successfully reset to pristine condition.


### Data Populating : 

In [30]:
import os
import re
from IPython import get_ipython
import oracledb


def load_entire_customer_orders_schema():
    base_dir = r"E:\Study\Github\Repositories\Learn\SQL\PL SQL\PL SQL By Prashant\Sample Data\db-sample-schemas-23.3\customer_orders"

    file_create = os.path.join(base_dir, "co_create.sql")
    file_populate = os.path.join(base_dir, "co_populate.sql")

    ip = get_ipython()
    if ip is None:
        return

    # Securely retrieve the active underlying driver connection cursor from memory
    plsql_magic = ip.magics_manager.magics['cell'].get('plsql')
    if plsql_magic is None:
        print("❌ Active session not found. Please run connect_oracle() first.")
        return

    try:
        closure_vars = plsql_magic.__closure__
        cursor = next(c.cell_contents for c in closure_vars if isinstance(
            c.cell_contents, oracledb.Cursor))
        connection = cursor.connection
    except Exception:
        print("❌ Memory context binding failure. Re-execute connect_oracle().")
        return

    print("⏳ Database connection verified. Initializing master data pipeline...")

    # --- PHASE 1: EXECUTE STRUCTURES (co_create.sql) ---
    if os.path.exists(file_create):
        print("📦 Processing Structure definitions: co_create.sql...")
        with open(file_create, "r", encoding="utf-8") as f:
            content = f.read()

        # Clean out parameters and all variants of command-line comment strings
        clean_lines = []
        for line in content.splitlines():
            line_upper = line.strip().upper()
            if not line.strip() or any(line_upper.startswith(prefix) for prefix in ("SET", "PROMPT", "ACCEPT", "SHOW", "EXIT", "REM")):
                continue
            line_clean = re.sub(r'--.*$', '', line)
            clean_lines.append(line_clean)

        processed = "\n".join(clean_lines)
        statements = [s.strip() for s in processed.split(";") if s.strip()]

        success_count = 0
        for stmt in statements:
            if stmt.endswith('/'):
                stmt = stmt[:-1].strip()
            if not stmt:
                continue
            try:
                cursor.execute(stmt)
                success_count += 1
            except oracledb.DatabaseError as e:
                error, = e.args
                # Suppress table drop warnings if they don't exist yet
                if error.code not in (942, 1432, 2289, 2449):
                    print(
                        f"⚠️ Skipped line: {stmt[:40]}... | Error: {error.message}")

        connection.commit()
        print(
            f"✅ Successfully initialized {success_count} structural table allocations.")

    # --- PHASE 2: EXECUTE BLOCKS (co_populate.sql) ---
    if os.path.exists(file_populate):
        print("\n📦 Processing Content Blocks: co_populate.sql...")
        with open(file_populate, "r", encoding="utf-8") as f:
            content = f.read()

        clean_lines = []
        for line in content.splitlines():
            line_upper = line.strip().upper()
            if any(line_upper.startswith(prefix) for prefix in ("SET", "PROMPT", "ACCEPT", "SHOW", "EXIT", "REM", "ALTER SESSION", "COMMIT")):
                continue
            line_clean = re.sub(r'--.*$', '', line)
            clean_lines.append(line_clean)

        processed_data = "\n".join(clean_lines)

        # STABLE BATCH FIX: Split precisely on the client batch delimiter forward slash boundary
        # This keeps DECLARE and BEGIN blocks perfectly glued together as single strings!
        blocks = re.split(r'\n\s*/\s*\n', processed_data)

        success_blocks = 0
        print("⏳ Injecting multi-line row execution arrays into cloud instance...")

        for block in blocks:
            block_clean = block.strip()
            if block_clean.endswith('/'):
                block_clean = block_clean[:-1].strip()
            if not block_clean:
                continue

            try:
                cursor.execute(block_clean)
                success_blocks += 1
            except oracledb.DatabaseError as e:
                error, = e.args
                # Ignore minor index modifications if tables reset smoothly
                if error.code not in (942, 1432, 2289):
                    print(
                        f"❌ Block Ingestion Aborted: {block_clean[:60]}... \n↳ Error: {error.message}\n")

        connection.commit()
        print(
            f"✅ Successfully written transactional data rows: {success_blocks} items.")
        print("🎉 Database optimization complete. All records populated safely!")


load_entire_customer_orders_schema()

⏳ Database connection verified. Initializing master data pipeline...
📦 Processing Structure definitions: co_create.sql...
✅ Successfully initialized 120 structural table allocations.

📦 Processing Content Blocks: co_populate.sql...
⏳ Injecting multi-line row execution arrays into cloud instance...
❌ Block Ingestion Aborted: ALTER TABLE products
  MODIFY product_id
  GENERATED BY DEFA... 
↳ Error: ORA-03405: End of query reached; no additional text should follow.
Help: https://docs.oracle.com/error-help/db/ora-03405/

✅ Successfully written transactional data rows: 7 items.
🎉 Database optimization complete. All records populated safely!


### Record Verification : 

In [31]:
%%plsql
DECLARE
    v_products  NUMBER;
    v_inventory NUMBER;
    v_orders    NUMBER;
BEGIN
    SELECT COUNT(*) INTO v_products FROM products;
    SELECT COUNT(*) INTO v_inventory FROM inventory;
    SELECT COUNT(*) INTO v_orders FROM orders;
    
    DBMS_OUTPUT.PUT_LINE('📦 Verification Success! Total Products Found: ' || v_products);
    DBMS_OUTPUT.PUT_LINE('📦 Verification Success! Total Inventory Records: ' || v_inventory);
    DBMS_OUTPUT.PUT_LINE('📊 Verification Success! Total Active Orders Stored: ' || v_orders);
END;


📦 Verification Success! Total Products Found: 46
📦 Verification Success! Total Inventory Records: 566
📊 Verification Success! Total Active Orders Stored: 1950


In [32]:
%%plsql
DECLARE
    v_count NUMBER;
BEGIN
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
    DBMS_OUTPUT.PUT_LINE(RPAD('TABLE NAME', 30) || ' | ' || 'LIVE ROW COUNT');
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
    
    -- Loop through every user table owned by your active schema profile
    FOR t IN (SELECT table_name FROM user_tables ORDER BY table_name) LOOP
        BEGIN
            -- Construct and evaluate dynamic SQL to pull the exact current count
            EXECUTE IMMEDIATE 'SELECT COUNT(*) FROM "' || t.table_name || '"' INTO v_count;
            
            -- Format and output the layout nicely to the terminal block console
            DBMS_OUTPUT.PUT_LINE(RPAD(t.table_name, 30) || ' | ' || TO_CHAR(v_count, '999,999'));
        EXCEPTION
            WHEN OTHERS THEN
                DBMS_OUTPUT.PUT_LINE(RPAD(t.table_name, 30) || ' | ERROR: ' || SQLERRM);
        END;
    END LOOP;
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
END;


-------------------------------------------
TABLE NAME                     | LIVE ROW COUNT
-------------------------------------------
CUSTOMERS                      |      392
INVENTORY                      |      566
ORDERS                         |    1,950
ORDER_ITEMS                    |    3,914
PRODUCTS                       |       46
SHIPMENTS                      |    1,892
STORES                         |       23
-------------------------------------------


## Human Resources Table :

### Schema Formatting :

In [52]:
%%plsql
DECLARE
    CURSOR c_hr_tables IS
        SELECT table_name 
        FROM user_tables 
        WHERE table_name IN ('EMPLOYEES', 'DEPARTMENTS', 'JOBS', 'JOB_HISTORY', 'LOCATIONS', 'COUNTRIES', 'REGIONS');
        
    CURSOR c_hr_seqs IS
        SELECT sequence_name 
        FROM user_sequences 
        WHERE sequence_name IN ('LOCATIONS_SEQ', 'DEPARTMENTS_SEQ', 'EMPLOYEES_SEQ');
BEGIN
    -- Drop tables with cascade rules to break foreign-key dependencies
    FOR r_tab IN c_hr_tables LOOP
        BEGIN
            EXECUTE IMMEDIATE 'DROP TABLE ' || r_tab.table_name || ' CASCADE CONSTRAINTS';
            DBMS_OUTPUT.PUT_LINE('🧹 Dropped table structure: ' || r_tab.table_name);
        EXCEPTION WHEN OTHERS THEN NULL;
        END;
    END LOOP;
    
    -- Clear background auto-increment tracking sequences
    FOR r_seq IN c_hr_seqs LOOP
        BEGIN
            EXECUTE IMMEDIATE 'DROP SEQUENCE ' || r_seq.sequence_name;
            DBMS_OUTPUT.PUT_LINE('🧹 Cleared duplicate tracking sequence: ' || r_seq.sequence_name);
        EXCEPTION WHEN OTHERS THEN NULL;
        END;
    END LOOP;
    
    DBMS_OUTPUT.PUT_LINE('✅ HR workspace is completely reset and pristine.');
END;


🧹 Dropped table structure: COUNTRIES
🧹 Dropped table structure: DEPARTMENTS
🧹 Dropped table structure: EMPLOYEES
🧹 Dropped table structure: JOBS
🧹 Dropped table structure: JOB_HISTORY
🧹 Dropped table structure: LOCATIONS
🧹 Dropped table structure: REGIONS
🧹 Cleared duplicate tracking sequence: DEPARTMENTS_SEQ
🧹 Cleared duplicate tracking sequence: EMPLOYEES_SEQ
🧹 Cleared duplicate tracking sequence: LOCATIONS_SEQ
✅ HR workspace is completely reset and pristine.


### HR Data Populating : 

In [53]:
import os
import re
from IPython import get_ipython
import oracledb


def load_entire_human_resources_schema_perfectly():
    base_dir = r"E:\Study\Github\Repositories\Learn\SQL\PL SQL\PL SQL By Prashant\Sample Data\db-sample-schemas-23.3\human_resources"

    file_create = os.path.join(base_dir, "hr_create.sql")
    file_populate = os.path.join(base_dir, "hr_populate.sql")

    ip = get_ipython()
    if ip is None:
        return

    plsql_magic = ip.magics_manager.magics['cell'].get('plsql')
    if plsql_magic is None:
        print("❌ Active session not found. Please run connect_oracle() first.")
        return

    try:
        closure_vars = plsql_magic.__closure__
        cursor = next(c.cell_contents for c in closure_vars if isinstance(
            c.cell_contents, oracledb.Cursor))
        connection = cursor.connection
    except Exception:
        print("❌ Memory context binding failure. Re-execute connect_oracle().")
        return

    print("⏳ Database connection verified. Initializing master HR data pipeline...")

    # --- PHASE 1: EXECUTE STRUCTURES (hr_create.sql) ---
    if os.path.exists(file_create):
        print("📦 Processing Structure definitions: hr_create.sql...")
        with open(file_create, "r", encoding="utf-8") as f:
            content = f.read()

        clean_lines = []
        for line in content.splitlines():
            line_upper = line.strip().upper()
            if not line.strip() or any(line_upper.startswith(prefix) for prefix in ("SET", "PROMPT", "ACCEPT", "SHOW", "EXIT", "REM")):
                continue
            clean_lines.append(line)

        processed_script = "\n".join(clean_lines)

        statements = []
        in_quote = False
        current_stmt = []
        for char in processed_script:
            if char == "'":
                in_quote = not in_quote
            if char == ";" and not in_quote:
                statements.append("".join(current_stmt).strip())
                current_stmt = []
            else:
                current_stmt.append(char)
        if current_stmt:
            statements.append("".join(current_stmt).strip())

        success_count = 0
        for stmt in statements:
            if stmt.endswith('/'):
                stmt = stmt[:-1].strip()
            if not stmt:
                continue
            try:
                cursor.execute(stmt)
                success_count += 1
            except oracledb.DatabaseError as e:
                error, = e.args
                if error.code not in (942, 1432, 2289, 2449):
                    print(
                        f"⚠️ Skipped structure line: {stmt[:50]}... | Error: {error.message}")

        connection.commit()
        print(
            f"✅ Successfully initialized {success_count} structural table allocations.")

    # --- PHASE 2: EXECUTE DATA INJECTION (hr_populate.sql) ---
    if os.path.exists(file_populate):
        print("\n📦 Processing Content Blocks: hr_populate.sql...")
        with open(file_populate, "r", encoding="utf-8") as f:
            content = f.read()

        # Step A: Safely execute the manual constraint suspension command ahead of the loop blocks
        print("🔓 Temporarily disabling circular manager constraints...")
        try:
            cursor.execute(
                "ALTER TABLE departments DISABLE CONSTRAINT dept_mgr_fk")
        except oracledb.DatabaseError as e:
            pass  # Keep moving if the table constraint is altered already

        # Step B: Parse out the remaining data blocks uniformly
        clean_lines = []
        for line in content.splitlines():
            line_upper = line.strip().upper()
            if any(line_upper.startswith(prefix) for prefix in ("SET", "PROMPT", "ACCEPT", "SHOW", "EXIT", "REM", "ALTER SESSION", "COMMIT", "ALTER TABLE")):
                continue
            line_clean = re.sub(r'--.*$', '', line)
            clean_lines.append(line_clean)

        processed_data = "\n".join(clean_lines)

        # Remove any lingering split fragments left behind by multi-line format structures
        processed_data = re.sub(
            r'(?:DISABLE|ENABLE)\s+CONSTRAINT\s+dept_mgr_fk\s*;', '', processed_data, flags=re.IGNORECASE)

        # Split precisely on the client batch delimiter forward-slash operator (\n/\n)
        blocks = re.split(r'\n\s*/\s*\n', processed_data)

        success_blocks = 0
        print("⏳ Injecting multi-line row execution arrays into cloud instance...")
        for block in blocks:
            block_clean = block.strip()
            if block_clean.endswith('/'):
                block_clean = block_clean[:-1].strip()
            if not block_clean:
                continue

            try:
                cursor.execute(block_clean)
                success_blocks += 1
            except oracledb.DatabaseError as e:
                error, = e.args
                if error.code not in (942, 1432, 2289):
                    print(
                        f"❌ Block Ingestion Aborted: {block_clean[:50]}... \n↳ Error: {error.message}\n")

        # Step C: Re-enable the constraint once all data rows exist securely in your schema views
        print("🔒 Re-enabling operational relational validation constraints...")
        try:
            cursor.execute(
                "ALTER TABLE departments ENABLE CONSTRAINT dept_mgr_fk")
        except oracledb.DatabaseError as e:
            print(
                f"⚠️ Warning finishing schema validation lock: {e.args[0].message}")

        connection.commit()
        print(
            f"✅ Successfully written transactional data rows: {success_blocks} items.")
        print("🎉 Entire HR Schema dataset successfully loaded and optimized!")


# Execute the final fix script
load_entire_human_resources_schema_perfectly()

⏳ Database connection verified. Initializing master HR data pipeline...
📦 Processing Structure definitions: hr_create.sql...
✅ Successfully initialized 78 structural table allocations.

📦 Processing Content Blocks: hr_populate.sql...
🔓 Temporarily disabling circular manager constraints...
⏳ Injecting multi-line row execution arrays into cloud instance...
🔒 Re-enabling operational relational validation constraints...
✅ Successfully written transactional data rows: 7 items.
🎉 Entire HR Schema dataset successfully loaded and optimized!
